# 11 — Barrier Options

- **Analytic** (Reiner-Rubinstein) — single barrier
- **Double barrier** (knock-out / knock-in)
- **Finite-difference** (FD)
- **Monte Carlo**
- Barrier types: down-and-out, down-and-in, up-and-out, up-and-in
- AD Greeks through the barrier

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
QL = load_quantlib()

In [ ]:
from ql_jax.engines.analytic.barrier import barrier_price
from ql_jax.engines.analytic.double_barrier import double_barrier_price, double_knock_in_price
from ql_jax.engines.fd.barrier import fd_barrier_price
from ql_jax.engines.mc.barrier import mc_barrier_bs

S, K, T, r, q, sigma = 100.0, 100.0, 1.0, 0.05, 0.02, 0.25
BARRIER_LO = 80.0   # down barrier
BARRIER_HI = 120.0  # up barrier
REBATE = 0.0

---
## 1. QuantLib Reference

In [ ]:
today = QL.Date(15, 6, 2024)
QL.Settings.instance().evaluationDate = today
maturity = QL.Date(15, 6, 2025)

dc = QL.Actual365Fixed()
cal = QL.NullCalendar()

spot_h = QL.QuoteHandle(QL.SimpleQuote(S))
r_ts   = QL.YieldTermStructureHandle(QL.FlatForward(today, r, dc))
q_ts   = QL.YieldTermStructureHandle(QL.FlatForward(today, q, dc))
vol_ts = QL.BlackVolTermStructureHandle(QL.BlackConstantVol(today, cal, sigma, dc))
bsm    = QL.BlackScholesMertonProcess(spot_h, q_ts, r_ts, vol_ts)

payoff = QL.PlainVanillaPayoff(QL.Option.Call, K)
exercise = QL.EuropeanExercise(maturity)

# Down-and-out call
dao = QL.BarrierOption(QL.Barrier.DownOut, BARRIER_LO, REBATE, payoff, exercise)
dao.setPricingEngine(QL.AnalyticBarrierEngine(bsm))
ql_dao = dao.NPV()

# Up-and-out call
uao = QL.BarrierOption(QL.Barrier.UpOut, BARRIER_HI, REBATE, payoff, exercise)
uao.setPricingEngine(QL.AnalyticBarrierEngine(bsm))
ql_uao = uao.NPV()

# Down-and-in call
dai = QL.BarrierOption(QL.Barrier.DownIn, BARRIER_LO, REBATE, payoff, exercise)
dai.setPricingEngine(QL.AnalyticBarrierEngine(bsm))
ql_dai = dai.NPV()

print(f"QL down-and-out call : {ql_dao:.6f}")
print(f"QL up-and-out call   : {ql_uao:.6f}")
print(f"QL down-and-in call  : {ql_dai:.6f}")

---
## 2. ql-jax Analytic

In [ ]:
jax_dao = float(barrier_price(S, K, T, r, q, sigma, BARRIER_LO,
                               rebate=REBATE, option_type='call', barrier_type='down_and_out'))
jax_uao = float(barrier_price(S, K, T, r, q, sigma, BARRIER_HI,
                               rebate=REBATE, option_type='call', barrier_type='up_and_out'))
jax_dai = float(barrier_price(S, K, T, r, q, sigma, BARRIER_LO,
                               rebate=REBATE, option_type='call', barrier_type='down_and_in'))
jax_uai = float(barrier_price(S, K, T, r, q, sigma, BARRIER_HI,
                               rebate=REBATE, option_type='call', barrier_type='up_and_in'))

rows = [
    ("Down-and-Out", ql_dao, jax_dao),
    ("Up-and-Out",   ql_uao, jax_uao),
    ("Down-and-In",  ql_dai, jax_dai),
    ("Up-and-In",    None,   jax_uai),
]

print(f"\n{'Type':<20s} {'QL':>12s} {'ql-jax':>12s} {'Diff':>12s}")
print("-" * 60)
for label, ql_v, jax_v in rows:
    if ql_v is None:
        print(f"{label:<20s} {'—':>12s} {jax_v:>12.6f}")
    else:
        print(f"{label:<20s} {ql_v:>12.6f} {jax_v:>12.6f} {abs(ql_v-jax_v):>12.2e}")

---
## 3. In-Out Parity

For a given underlying option and barrier level:
$$V_{\text{knock-in}} + V_{\text{knock-out}} = V_{\text{vanilla}}$$

In [ ]:
from ql_jax.engines.analytic.european import bsm_price

vanilla = float(bsm_price(S, K, T, r, q, sigma, option_type=1))

down_sum = jax_dao + jax_dai
up_sum   = jax_uao + jax_uai

print(f"Vanilla call                  : {vanilla:.6f}")
print(f"Down-out + Down-in            : {down_sum:.6f}  (diff = {abs(vanilla - down_sum):.2e})")
print(f"Up-out + Up-in                : {up_sum:.6f}  (diff = {abs(vanilla - up_sum):.2e})")

---
## 4. Double Barrier

In [ ]:
jax_dko = float(double_barrier_price(S, K, T, r, q, sigma, BARRIER_LO, BARRIER_HI, option_type='call'))
jax_dki = float(double_knock_in_price(S, K, T, r, q, sigma, BARRIER_LO, BARRIER_HI, option_type='call'))

print(f"Double knock-out : {jax_dko:.6f}")
print(f"Double knock-in  : {jax_dki:.6f}")
print(f"DKO + DKI        : {jax_dko + jax_dki:.6f}  (vanilla = {vanilla:.6f})")

---
## 5. FD & MC Engines

In [ ]:
jax_fd = float(fd_barrier_price(S, K, T, r, q, sigma, BARRIER_LO,
                                 option_type=1, barrier_type='down_and_out',
                                 rebate=REBATE, n_space=400, n_time=400))

mc_val, mc_se = mc_barrier_bs(S, K, T, r, q, sigma,
                               option_type=1, barrier_type=2,   # DownOut=2
                               barrier=BARRIER_LO, rebate=REBATE,
                               n_paths=200_000, n_steps=500)
jax_mc, jax_se = float(mc_val), float(mc_se)

print(f"{'Engine':<20s} {'Price':>12s}")
print("-" * 35)
print(f"{'Analytic':<20s} {jax_dao:>12.6f}")
print(f"{'FD (400×400)':<20s} {jax_fd:>12.6f}")
print(f"{'MC (200k paths)':<20s} {jax_mc:>12.6f} ± {jax_se:.4f}")
print(f"{'QL analytic':<20s} {ql_dao:>12.6f}")

---
## 6. AD: Delta Near the Barrier

In [ ]:
import matplotlib.pyplot as plt

def dao_price(spot):
    return barrier_price(spot, K, T, r, q, sigma, BARRIER_LO,
                         rebate=0.0, option_type='call', barrier_type='down_and_out')

delta_fn = jax.grad(dao_price)
gamma_fn = jax.grad(jax.grad(dao_price))

spots = jnp.linspace(82, 130, 100)
prices = [float(dao_price(s)) for s in spots]
deltas = [float(delta_fn(s)) for s in spots]
gammas = [float(gamma_fn(s)) for s in spots]

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

ax1.plot(np.array(spots), prices, 'b-', linewidth=2)
ax1.axvline(BARRIER_LO, color='red', linestyle='--', alpha=0.7, label=f'Barrier={BARRIER_LO}')
ax1.set_xlabel('Spot'); ax1.set_ylabel('Price')
ax1.set_title('Down-and-Out Call Price'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(np.array(spots), deltas, 'g-', linewidth=2)
ax2.axvline(BARRIER_LO, color='red', linestyle='--', alpha=0.7)
ax2.set_xlabel('Spot'); ax2.set_ylabel('Delta')
ax2.set_title('Delta (AD)'); ax2.grid(True, alpha=0.3)

ax3.plot(np.array(spots), gammas, 'm-', linewidth=2)
ax3.axvline(BARRIER_LO, color='red', linestyle='--', alpha=0.7)
ax3.set_xlabel('Spot'); ax3.set_ylabel('Gamma')
ax3.set_title('Gamma (AD)'); ax3.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

---
## 7. Exercises

1. **Rebate pricing**: Set a non-zero rebate and verify in-out parity still holds.
2. **Vanna-Volga**: If available, compare the Vanna-Volga barrier price to the analytic engine.
3. **FD convergence**: Plot FD price vs grid resolution and measure convergence rate.

---
*End of Notebook 11*